In [53]:
import importlib
import user_class

importlib.reload(user_class)
from user_class import User

print(User.__module__)
print(User.__init__.__code__.co_varnames)


user_class
('self', 'user_id', 'bankroll', 'bet_history', 'pitch_index', 'table')


In [54]:
import os, time
import boto3
from datetime import datetime, timezone
from decimal import Decimal
from botocore.exceptions import ClientError
from fastapi import HTTPException

In [55]:
DEFAULT_BANKROLL = float(os.getenv("DEFAULT_BANKROLL", "100"))

# create function to return DynamoDB table storing user session information
def get_table():
    dynamodb = boto3.resource("dynamodb", region_name=os.getenv("AWS_REGION", "us-east-1"))
    table = dynamodb.Table(os.getenv("SESSIONS_TABLE", "PitchBettingSessions"))
    return table

In [56]:
def create_user(user_id, table):
    # create a new user with the default bankroll and empty bet history
    user = User(
        user_id=user_id,
        bankroll=DEFAULT_BANKROLL,
        bet_history=[],
        pitch_index=0,
        table=table
    )

    user.save_to_db()

    return user

In [57]:
print(User.__module__)
print(User.__init__.__code__.co_varnames)


user_class
('self', 'user_id', 'bankroll', 'bet_history', 'pitch_index', 'table')


In [58]:
def get_user(user_id):
    table = get_table()
    resp = table.get_item(Key={"user_id": user_id})
    if "Item" in resp:
        item = resp["Item"]

        # create a User object from the DynamoDB item
        user = User(
            user_id=user_id,
            bankroll=float(item["bankroll"]),
            bet_history=item["bet_history"],
            pitch_index=int(item["pitch_index"]),
            table=table
        )

    # otherwise, create a new user with the default bankroll and empty bet history
    else:
        user = create_user(user_id, table)

    return user

user = get_user("topher-test2")

In [63]:
user.advance_pitch()

In [64]:
user.get_pitch_index()

3

In [75]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [77]:
# load data frame from s3

    # df = pd.read_parquet("heldout_games.parquet")
import pandas as pd
import os
import re
import pickle
import joblib
import utils
importlib.reload(utils)
import wandb
import src

api = wandb.Api()
s3_uri = "s3://statcast-mlb-raw/pitches/heldout_games.parquet"
df = pd.read_parquet(s3_uri)
df = df.sort_values(["game_pk", "at_bat_number", "pitch_number"])



    # Login to W&B (uses WANDB_API_KEY env variable)
    # wandb.login(key="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")

  

ENTITY = 'chris-r-thompson1212-university-of-denver'
PROJECT = "money-ball"
pipeline = utils.load_production_model(ENTITY, PROJECT)
# get a single row from df
row = df.iloc[0]

input_df = pd.DataFrame([row])
output_df =pipeline[:-1].transform(input_df)

print(output_df)
# input row into model.transform

# check output of model.transform


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/topherthompson/.netrc
wandb:   1 of 1 files downloaded.  


AttributeError: 'tuple' object has no attribute 'transform'